# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WingManOO7/ML-INTER/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

**Optional stretch, completed.** The card for this notebook was retired on 2026-07-13 and its core moved into the graded notebooks, but the capstone rewards the full-depth version.

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

The optional stretch notebook, done on the same ten development decision dates (2026-01-05 → 2026-05-11)
and with the same shared module (`work/scripts/search_queue.py`) as ML-09, ML-10 and the capstone. June 2026
stays sealed.

In [1]:
import os, sys, json, subprocess, importlib.util, urllib.request, warnings
from pathlib import Path
warnings.filterwarnings('ignore', category=FutureWarning)

if importlib.util.find_spec('duckdb') is None:      # Colab ships duckdb; this covers fresh local envs
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'])

import numpy as np
import pandas as pd
import sklearn

# the shared module: from the repo when run inside it, else fetched from the repo (Colab opened from GitHub)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'work' / 'notebooks').is_dir()), None)
if ROOT is not None:
    sys.path.insert(0, str(ROOT / 'work' / 'scripts'))
else:
    ROOT = Path.cwd()
    urllib.request.urlretrieve('https://raw.githubusercontent.com/WingManOO7/ML-INTER/main/work/scripts/search_queue.py',
                               'search_queue.py')
import search_queue as sq

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste a token into a cell.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass('Hugging Face READ token: ')

OUT_DIR = ROOT / 'work' / 'outputs'
CACHE = OUT_DIR / 'cache'            # parquet frames, gitignored: reruns skip the warehouse scans
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEV_DATES = sq.decision_dates('2026-01-05', '2026-05-11')     # every second Monday; outcomes end 2026-05-24
DEV_MONTHS = sq.months_needed(DEV_DATES)
assert '2026-06' not in DEV_MONTHS, 'June 2026 is the sealed month: development never opens it'

con = sq.open_warehouse(DEV_MONTHS, HF_TOKEN)
frames = sq.build_frames(con, DEV_DATES, cache_dir=CACHE)
dev = frames[frames['label_ok']].reset_index(drop=True)
SEED = 42

print(f'{len(DEV_DATES)} decision dates {DEV_DATES[0].date()} -> {DEV_DATES[-1].date()} | months {DEV_MONTHS[0]} -> {DEV_MONTHS[-1]}')
print(f'{len(frames):,} page-date rows; {len(dev):,} with a complete outcome window; '
      f'{dev.client_hash_id.nunique()} clients | pandas {pd.__version__}, scikit-learn {sklearn.__version__}')

10 decision dates 2026-01-05 -> 2026-05-11 | months 2025-12 -> 2026-05
659,482 page-date rows; 654,186 with a complete outcome window; 50 clients | pandas 2.3.3, scikit-learn 1.7.2


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The feature vector is built in SQL plus a little pandas by `search_queue.build_frame(con, T)`:

1. **One DuckDB aggregation per decision date** over the 42 days around T: the 14 days before it (the
   feature window), the 14 before that (look-back) and the 14 from T (the outcome, labels only). Every
   sum is filtered with `gsc_data_available IS TRUE`.
2. **Eligibility:** ≥ 100 impressions before T, tracked from the window's first day, and a valid
   position.
3. **Engineered features:** logs, ratios, peak-day share and the site-level slide.
4. **Fills:** a week with no impressions gets 0 (a real zero, because the page was tracked).
   `pos_pre` is never filled: pages without a valid position are dropped.
5. **No categorical features.** `content_type` exists in `dim_content` but isn't used, and the IDs are
   pseudonyms used for grouping only.

In [2]:
FEATURES = sq.CONTRACT_FEATURES + sq.EXTRA_FEATURES
sq.assert_no_outcome_features(FEATURES)
vec = dev[['decision_date', 'client_hash_id', 'content_hash_id'] + FEATURES]
print(f'feature vector: {len(vec):,} page-dates x {len(FEATURES)} features; missing values per feature: '
      f'{vec[FEATURES].isna().sum().to_dict()}\n')
vec[FEATURES].describe(percentiles=[0.01, 0.5, 0.99]).T.round(3)

feature vector: 654,186 page-dates x 8 features; missing values per feature: {'log_imp_pre': 0, 'pos_pre': 0, 'active_days_pre': 0, 'momentum_pre': 0, 'age_days_at_T': 0, 'momentum_4w': 0, 'peak_day_share_pre': 0, 'site_momentum': 0}



,count,mean,std,min,1%,50%,99%,max
log_imp_pre,654186.0,6.412,1.228,4.615,4.644,6.240,9.602,12.916
pos_pre,654186.0,12.395,12.471,0.006,0.957,7.629,61.951,142.294
active_days_pre,654186.0,13.810,0.906,1.000,9.000,14.000,14.000,14.000
momentum_pre,654186.0,0.009,0.696,-7.616,-1.744,0.000,2.060,9.593
age_days_at_T,654186.0,190.991,119.242,3.000,13.000,181.000,461.000,535.000
momentum_4w,654186.0,0.333,1.273,-6.390,-1.370,0.092,6.422,10.450
peak_day_share_pre,654186.0,0.138,0.061,0.075,0.085,0.122,0.393,1.000
site_momentum,654186.0,0.006,0.174,-2.833,-0.346,0.018,0.408,2.340


**654,186 page-dates × 8 features, with zero missing values** after the eligibility rules. The two impression features are already on a log scale, and every page carries at least 100 impressions. Two things to know before modelling:
- **`momentum_pre` and `momentum_4w` have long tails** (−7.6 to +9.6), from pages that went to almost nothing or appeared from almost nothing.
- **`pos_pre` has a minimum of 0.006.** A position below 1 can't come from Search Console's scale; it affects 1.1% of page-dates (ML-06) and is left as is, because a tree or logistic model is barely moved by it and dropping those rows would hide a real data-quality issue.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing / fill | Available when? |
|---|---|---|---|
| `log_imp_pre` | log(1 + impressions over the 14 days before T) | never missing (eligibility needs ≥ 100) | at T: every summed day is before T |
| `pos_pre` | impression-weighted average position, position 0 ("no data") skipped | pages with no valid position are **dropped**, not filled | at T |
| `active_days_pre` | days with impressions out of 14 | never missing | at T |
| `momentum_pre` | log((week 2 + 1) / (week 1 + 1)) | a week without impressions counts as 0 (+1 smoothing) | at T: both weeks end before T |
| `age_days_at_T` | T − `content_created_date` | pages without a creation date are dropped (none were) | at T: creation precedes T and never moves |
| `momentum_4w` | log((14 days before T + 1) / (the 14 days before those + 1)) | an empty look-back counts as 0 | at T |
| `peak_day_share_pre` | the largest single day ÷ the 14-day total | never missing | at T |
| `site_momentum` | the same log slide for the whole site (sum over its eligible pages) | never missing | at T: other pages' past only |

**Proof, not a promise.** The cell below recomputes three features for one decision date **straight from the
raw daily rows**, keeping only days strictly before T, and checks they equal the frame's values page for
page.

In [3]:
T = pd.Timestamp('2026-03-16')
raw = con.sql(f'''
    SELECT content_hash_id,
           SUM(gsc_impressions) FILTER (WHERE report_date <  DATE '2026-03-09') AS w1,
           SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-09') AS w2,
           COUNT(*) AS active_days
    FROM daily
    WHERE gsc_data_available IS TRUE AND report_date >= DATE '2026-03-02' AND report_date < DATE '{T.date()}'
    GROUP BY content_hash_id
''').df()
check = dev[dev['decision_date'] == T].merge(raw, on='content_hash_id', how='left')
check[['w1', 'w2']] = check[['w1', 'w2']].fillna(0)
same = {
    'momentum_pre': np.allclose(check['momentum_pre'], np.log((check['w2'] + 1) / (check['w1'] + 1))),
    'log_imp_pre': np.allclose(check['log_imp_pre'], np.log1p(check['w1'] + check['w2'])),
    'active_days_pre': (check['active_days_pre'] == check['active_days']).all(),
}
print(f'{len(check):,} pages at {T.date()}: features recomputed from rows dated before T only ->', same)
assert all(same.values())

74,542 pages at 2026-03-16: features recomputed from rows dated before T only -> {'momentum_pre': True, 'log_imp_pre': True, 'active_days_pre': np.True_}


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Four attacks. Each is a rolling-origin, client-grouped backtest, so an attack only "works" if it would fool
an honest evaluation:

- **A. Label-derived column.** The module refuses outcome columns **by name**. A renamed derivation
  (`log_imp_out`) slips past any name check, so I measure what it would do. That gap is why the tests pin
  the windows themselves, and why section 2 recomputes features from raw rows.
- **B. A future window hiding in a snapshot.** `content_updated_date` looks like a harmless page
  attribute, but it is a July export: for most pages it records an update *after* T. Turned into "days
  since update at T", does it inflate the score?
- **C. Shuffled labels.** A sanity floor: the score must fall to chance.
- **D. Product flags and IDs.** Checked against the schema and the feature list.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
make_lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
def auc_p50(frame, features, label='declined_robust'):
    cells = sq.cell_metrics(sq.rolling_backtest(frame, {'m': (features, make_lr)}, label=label), {'m': 'p_m'})
    return round(cells['AUC'].mean(), 3), round(cells['P@50'].mean(), 3)

hunt = {'honest (5 contract features)': auc_p50(frames, sq.CONTRACT_FEATURES)}

# A. label-derived column: the guard refuses the outcome column by NAME ...
try:
    sq.assert_no_outcome_features(sq.CONTRACT_FEATURES + ['imp_out'])
except AssertionError as e:
    print('A. guard:', e)
# ... but a renamed derivation slips past any name check, so measure what it would do
leaky = frames.assign(log_imp_out=np.log1p(frames['imp_out']))
hunt['A. + log(outcome impressions), renamed'] = auc_p50(leaky, sq.CONTRACT_FEATURES + ['log_imp_out'])

# B. the snapshot update date as a feature
upd = con.sql('SELECT content_hash_id, content_updated_date FROM dim_content').df()
snap = frames.merge(upd, on='content_hash_id', how='left')
snap['days_since_update_snapshot'] = (snap['decision_date'] - pd.to_datetime(snap['content_updated_date'])).dt.days
print(f"B. share of page-dates whose 'last update' lies AFTER T: {(snap['days_since_update_snapshot'] < 0).mean():.3f}")
hunt['B. + days since update (July snapshot)'] = auc_p50(snap, sq.CONTRACT_FEATURES + ['days_since_update_snapshot'])

# C. shuffled labels
shuffled = frames.assign(declined_robust=np.random.RandomState(SEED).permutation(frames['declined_robust'].to_numpy()))
hunt['C. shuffled labels'] = auc_p50(shuffled, sq.CONTRACT_FEATURES)

# D. product flags and IDs
flags = {'health_score', 'priority_score', 'action_type', 'needs_ctr_fix', 'is_quick_win', 'refresh_tier'}
schema = set(con.sql('DESCRIBE daily').df()['column_name']) | set(con.sql('DESCRIBE dim_content').df()['column_name'])
ids = {c for c in schema if c.endswith('_hash_id')}
print('D. product flags in the release:', sorted(flags & schema) or 'none', '| ID columns used as features:', sorted(ids & set(FEATURES)) or 'none')
print()
pd.DataFrame(hunt, index=['AUC', 'precision@50']).T

A. guard: outcome-window columns cannot be features: ['imp_out']


B. share of page-dates whose 'last update' lies AFTER T: 0.870


D. product flags in the release: none | ID columns used as features: none



,AUC,precision@50
honest (5 contract features),0.653,0.810
"A. + log(outcome impressions), renamed",0.994,0.997
B. + days since update (July snapshot),0.658,0.817
C. shuffled labels,0.501,0.340


**What each attack did:**
- **A. The renamed outcome column** lifts AUC from **0.653 to 0.994** and precision@50 from 0.810 to 0.997. That is the signature of a leak. It also shows the limit of a name-based guard: the defence that actually holds is window-level tests plus recomputing features from raw rows (section 2).
- **B. The July update date** barely moves the score (AUC 0.653 → 0.658). It isn't a strong leak here, but for **87%** of page-dates its "last update" lies *after* T: it is future information, and a model that leaned on it would break in production. It stays excluded on principle, not because of the score.
- **C. Shuffled labels** give AUC 0.501 and precision@50 0.340 against a base rate of about 0.34. The pipeline doesn't manufacture signal.
- **D. Product flags and IDs:** none of FlyRank's decision flags exist in the release, and no ID column is a feature.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

The cell below accounts for **every column in the two tables I touch**, so nothing is silently left out.

In [5]:
WHY = {
    'report_date': 'context: assigns rows to windows', 'client_hash_id': 'pseudonym: grouping and splitting only',
    'content_hash_id': 'pseudonym: join key only', 'month': 'partition key', 'gsc_data_available': 'row filter (IS TRUE)',
    'gsc_impressions': 'USED: features (before T) and label (after T)', 'gsc_sum_position': 'USED: position feature',
    'gsc_avg_position': 'USED: 0 = no data, skipped', 'gsc_clicks': 'excluded: visibility label; CTR is another lane',
    'client_has_gsc': 'excluded: client attribute, near-constant within the slice',
    'client_has_ga4': 'excluded: client attribute', 'ga4_data_available': 'excluded: GA4 not used; NULL on many rows',
    'content_created_date': 'USED: page age at T', 'content_updated_date': 'excluded: July snapshot, after T for most pages',
    'last_optimized_date': 'excluded: snapshot + a decision someone already made', 'is_published': 'excluded: snapshot status',
    'is_deleted': 'excluded: snapshot status', 'keyword_hash_id': 'pseudonym: grouping only', 'url_hash_id': 'pseudonym: grouping only',
    'search_volume': 'excluded: keyword tool, undated', 'cpc': 'excluded: keyword tool, undated', 'competition': 'excluded: keyword tool, undated',
    'competition_level': 'excluded: keyword tool, undated', 'keyword_created_date': 'excluded: not needed; keyword-level',
    'content_type': 'excluded: tested no categorical; kept for future work', 'main_intent': 'excluded: keyword-tool label, undated',
}
rows = []
for table in ['daily', 'dim_content']:
    for col in con.sql(f'DESCRIBE {table}').df()['column_name']:
        if col in WHY:
            why = WHY[col]
        elif col.startswith(('ga4_', 'sessions_', 'ai_')) or col == 'scroll_events':
            why = 'excluded: GA4 / AI / scroll measurement, outside the search question, patchy availability'
        else:
            why = 'excluded: content metadata not needed for a two-week visibility question'
        rows.append({'table': table, 'column': col, 'decision': why})
accounting = pd.DataFrame(rows)
print(accounting['decision'].str.split(':').str[0].value_counts().to_string(), '\n')
pd.set_option('display.max_rows', 80)
accounting

decision
excluded                44
pseudonym                6
USED                     4
context                  1
row filter (IS TRUE)     1
partition key            1 



,table,column,decision
0,daily,report_date,context: assigns rows to windows
1,daily,client_hash_id,pseudonym: grouping and splitting only
2,daily,content_hash_id,pseudonym: join key only
3,daily,client_has_gsc,"excluded: client attribute, near-constant with..."
4,daily,client_has_ga4,excluded: client attribute
5,daily,gsc_data_available,row filter (IS TRUE)
6,daily,ga4_data_available,excluded: GA4 not used; NULL on many rows
7,daily,gsc_impressions,USED: features (before T) and label (after T)
8,daily,gsc_clicks,excluded: visibility label; CTR is another lane
9,daily,gsc_sum_position,USED: position feature


**All 57 columns of the two tables are accounted for:**
- **4 are used:** impressions, the position sum and average, and the creation date.
- **9 serve as context:** 6 pseudonymous IDs, the report date, the `IS TRUE` flag and the partition key.
- **44 are excluded, each with its reason:**
  - the snapshot dates and statuses;
  - the GA4 / AI / scroll measurements (outside the search question, with patchy availability);
  - undated keyword-tool fields;
  - clicks;
  - content metadata a two-week visibility question doesn't need.

In [6]:
receipts = {'notebook': 'work/notebooks/w03_feature_leakage_check.ipynb', 'features': FEATURES,
            'raw_recompute_check': {k: bool(v) for k, v in same.items()},
            'leakage_hunt': {k: {'AUC': v[0], 'P@50': v[1]} for k, v in hunt.items()},
            'columns_accounted': int(len(accounting))}
(OUT_DIR / 'w03x_leakage_receipts.json').write_text(json.dumps(receipts, indent=2))
print('wrote', (OUT_DIR / 'w03x_leakage_receipts.json').relative_to(ROOT))

wrote work\outputs\w03x_leakage_receipts.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.